# FORUM correlations

This notebook treats FORUM as an outcome of a ranking policy and asks where it agrees or disagrees with other summaries of the same ranking. The main plots are ranking-level plots: a point is a policy × outcome × interface/depth combination, with FORUM and nDCG averaged over stories. This avoids treating millions of comment rows as independent observations.

Questions:

1. Do top-10 and full-list FORUM agree across all rankings?
2. Does FORUM agree with nDCG across all rankings?
3. Do feature-level FORUM values from the regression rankings align with regression coefficients for the audience and editor selectors?
4. How does ML feature attribution (SHAP) align with FORUM?

SHAP is calculated and persisted by Stage 9 from the exact frozen winners. This notebook only aligns the story-aggregated SHAP summaries with the canonical ranking-level FORUM values and produces the comparison plots.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent

PAPER1_TABLES = REPO_ROOT / 'model_output/selection_2025/paper1/reporting/tables'
FORUM_ANALYSIS_INFERENCE = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/inference'
FORUM_ANALYSIS_SCORES = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/policy_scores/policy_scores.parquet'
REGRESSION_ROOT = REPO_ROOT / 'model_output/selection_2025/regression/all'

from commentgap_analysis.presentation_labels import (
    ORDERING_DISPLAY_LABELS, model_selector_label,
    OUTCOME_DISPLAY_LABELS, OUTCOME_DISPLAY_ORDER,
    REPLY_DISPLAY_MARKERS,
)
from commentgap_analysis.forum_plotting import (
    plot_forum_ndcg,
    plot_ml_forum_shap,
    plot_regression_forum_coefficients,
    plot_top10_full_forum,
)

FEATURE_LABELS = {
    **OUTCOME_DISPLAY_LABELS,
    'lexdiv_length_adjusted': 'Lexical diversity (adjusted)',
}
CORRELATION_EXCLUDED_OUTCOMES = {'lexdiv_length_adjusted'}

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

## 1. Load canonical ranking-level products

`policy_summary.csv` supplies the canonical story-averaged FORUM estimates. The policy-score parquet is used to obtain the matching story-averaged nDCG values. We keep the primary sample and retain all ordering, reply-mode, pinning, and depth combinations.

In [ ]:
import json
from commentgap_analysis.forum_scores import FORUM_ANALYSIS_VERSION
for manifest_path in (
    FORUM_ANALYSIS_INFERENCE / 'inference_manifest.json',
    FORUM_ANALYSIS_SCORES.parent / 'policy_score_manifest.json',
):
    if not manifest_path.exists() or json.loads(manifest_path.read_text()).get('version') != FORUM_ANALYSIS_VERSION:
        raise RuntimeError('Stale pin/reply semantics. Rerun notebook 10 before notebook 12.')

policy_summary = pd.read_csv(FORUM_ANALYSIS_INFERENCE / 'policy_summary.csv')
policy_summary = policy_summary[policy_summary['sample'].eq('primary')].copy()
policy_summary['feature_label'] = policy_summary['outcome'].map(FEATURE_LABELS).fillna(policy_summary['outcome'])

score_columns = [
    'story_id', 'policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable',
    'outcome', 'outcome_role', 'depth', 'forum', 'ndcg'
]
policy_scores = pd.read_parquet(FORUM_ANALYSIS_SCORES, columns=score_columns)
# outcome_role distinguishes the AQuA secondary outcome within the score product;
# it is still included in the primary-sample policy summary and belongs in the
# all-feature comparison requested here.

ranking_metrics = (
    policy_scores
    .groupby(['policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable', 'outcome', 'depth'], as_index=False)
    .agg(forum=('forum', 'mean'), ndcg=('ndcg', 'mean'), n_stories=('story_id', 'nunique'))
)
ranking_metrics['feature_label'] = ranking_metrics['outcome'].map(FEATURE_LABELS).fillna(ranking_metrics['outcome'])
ranking_metrics['ordering_label'] = ranking_metrics['ordering'].map(ORDERING_DISPLAY_LABELS).fillna(ranking_metrics['ordering'])

# Check that the independent aggregation reproduces the published FORUM estimate.
check = policy_summary.merge(
    ranking_metrics,
    on=['policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable', 'outcome', 'depth'],
    suffixes=('_published', '_recomputed'),
)
check['absolute_difference'] = (check['estimate'] - check['forum']).abs()
display(check['absolute_difference'].describe().to_frame('published_vs_recomputed_abs_difference'))
display(ranking_metrics.head())
print(f'{len(ranking_metrics):,} ranking-level rows; {ranking_metrics["ordering"].nunique()} orderings; {ranking_metrics["outcome"].nunique()} features.')

## Correlation and outlier helpers

Pearson correlation captures linear agreement; Spearman correlation captures agreement in ranking order. The outlier tables use the absolute vertical distance from the reference line or, for FORUM–nDCG, a robust residual from a within-panel linear fit. Labels are intended as a short-list for inspection, not as an automatic substantive classification.

In [ ]:
def correlation_row(frame, x, y, group_name='overall'):
    d = frame[[x, y]].dropna()
    if len(d) < 3 or d[x].nunique() < 2 or d[y].nunique() < 2:
        return {'group': group_name, 'n': len(d), 'pearson_r': np.nan, 'spearman_rho': np.nan}
    return {
        'group': group_name,
        'n': len(d),
        'pearson_r': d[x].corr(d[y], method='pearson'),
        'spearman_rho': d[x].corr(d[y], method='spearman'),
    }

def correlations_by(frame, x, y, group_cols):
    group_by = group_cols[0] if len(group_cols) == 1 else group_cols
    rows = [correlation_row(frame.loc[idx], x, y, ' | '.join(map(str, key if isinstance(key, tuple) else (key,))))
             for key, idx in frame.groupby(group_by, dropna=False).groups.items()]
    return pd.DataFrame(rows).sort_values('group').reset_index(drop=True)



## 2. Top-10 versus full-list FORUM

Each point is a ranking condition and feature. Colour denotes the feature, marker shape denotes reply handling, and filled versus hollow markers denote pinning status. Large departures from the dashed identity line are the first FORUM-behaviour outliers to inspect.

In [ ]:
forum_wide = (
    ranking_metrics[~ranking_metrics['outcome'].isin(CORRELATION_EXCLUDED_OUTCOMES)]
    .pivot_table(index=['policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable', 'outcome', 'feature_label'],
                 columns='depth', values='forum', aggfunc='first')
    .reset_index()
)
forum_wide = forum_wide.rename(columns={'full': 'forum_full', 'top10': 'forum_top10'})
forum_wide = forum_wide.dropna(subset=['forum_full', 'forum_top10']).copy()
forum_wide['difference_top10_minus_full'] = forum_wide['forum_top10'] - forum_wide['forum_full']
forum_wide['abs_difference'] = forum_wide['difference_top10_minus_full'].abs()

display(correlation_row(forum_wide, 'forum_full', 'forum_top10'))
display(correlations_by(forum_wide, 'forum_full', 'forum_top10', ['feature_label']))

feature_order = [FEATURE_LABELS[outcome] for outcome in OUTCOME_DISPLAY_ORDER if FEATURE_LABELS[outcome] in forum_wide['feature_label'].unique()]
feature_palette = dict(zip(feature_order, sns.color_palette('tab10', len(feature_order))))
reply_markers = REPLY_DISPLAY_MARKERS
reply_labels = {'loose': 'Loose', 'trees': 'Trees', 'hidden': 'Hidden'}
plot_top10_full_forum(forum_wide, feature_order, feature_palette, reply_markers, reply_labels)

print('Largest top-10/full departures:')
display(forum_wide.nlargest(20, 'abs_difference')[['feature_label', 'ordering', 'reply_mode', 'pinned', 'forum_top10', 'forum_full', 'difference_top10_minus_full']])

## 3. FORUM versus nDCG

This uses the same ranking-level units as the previous section. Separate top-10 and full-list figures make it possible to see whether apparent agreement is specific to the displayed top 10 or persists over the full visible list.

In [ ]:
forum_ndcg = ranking_metrics[~ranking_metrics['outcome'].isin(CORRELATION_EXCLUDED_OUTCOMES)].copy()
display(correlation_row(forum_ndcg, 'forum', 'ndcg'))
display(correlations_by(forum_ndcg, 'forum', 'ndcg', ['depth', 'feature_label']))

feature_order = [FEATURE_LABELS[outcome] for outcome in OUTCOME_DISPLAY_ORDER if FEATURE_LABELS[outcome] in forum_ndcg['feature_label'].unique()]
feature_palette = dict(zip(feature_order, sns.color_palette('tab10', len(feature_order))))
reply_markers = REPLY_DISPLAY_MARKERS
reply_labels = {'loose': 'Loose', 'trees': 'Trees', 'hidden': 'Hidden'}
plot_forum_ndcg(forum_ndcg, feature_order, feature_palette, reply_markers, reply_labels)

print('Largest within-depth FORUM/nDCG residuals:')
outlier_parts = []
for depth, panel in forum_ndcg.groupby('depth'):
    panel = panel.copy()
    slope, intercept = np.polyfit(panel['forum'], panel['ndcg'], 1)
    panel['abs_residual'] = (panel['ndcg'] - (intercept + slope * panel['forum'])).abs()
    outlier_parts.append(panel)
display(pd.concat(outlier_parts).nlargest(20, 'abs_residual')[['depth', 'feature_label', 'ordering', 'reply_mode', 'pinned', 'forum', 'ndcg', 'abs_residual']])

## 4. Regression FORUM on feature versus coefficient

Here `x` is the story-averaged FORUM of a regression audience/editor ranking on the feature outcome, and `y` is the corresponding regression coefficient. The canonical interface is loose, unpinned; top-10 and full-list values are shown separately.

The selected FORUM/ranking analysis run no longer carries all 41 regression features. This diagnostic therefore plots only selected outcomes with matching coefficient rows and explicitly reports the remaining selected outcomes as unmatched.

In [ ]:
regression_coefficients = pd.read_csv(PAPER1_TABLES / 'regression_selector_associations.csv')
regression_coefficients = regression_coefficients[regression_coefficients['scope'].eq('all')].copy()

selector_map = {
    'audience': ('audience_log_odds', 'regression_audience'),
    'editor': ('curator_log_odds', 'regression_editor'),
}
coefficient_long = pd.concat([
    regression_coefficients[['term', 'feature', 'audience_log_odds']].rename(columns={'term': 'outcome', 'audience_log_odds': 'coefficient'}).assign(selector='audience', ordering='regression_audience'),
    regression_coefficients[['term', 'feature', 'curator_log_odds']].rename(columns={'term': 'outcome', 'curator_log_odds': 'coefficient'}).assign(selector='editor', ordering='regression_editor'),
], ignore_index=True)

regression_forum = ranking_metrics.query(
    "ordering in ['regression_audience', 'regression_editor'] and reply_mode == 'loose' and not pinned"
)[['ordering', 'outcome', 'depth', 'forum', 'feature_label']].copy()
regression_plot = regression_forum.merge(coefficient_long, on=['ordering', 'outcome'], how='inner')
regression_plot['feature_label'] = regression_plot['feature'].where(regression_plot['feature'].notna(), regression_plot['feature_label'])

def regression_feature_category(feature):
    feature = str(feature)
    lower = feature.lower()
    if feature.startswith('AQuA '):
        return 'AQuA'
    if 'earlier reply-to-root composition' in lower:
        return 'Timing / discussion\ncontext'
    if 'comment length' in lower or any(term in lower for term in ('url', 'reply', 'root')):
        return 'Comment form'
    if any(term in lower for term in ('length', 'lexical diversity', 'reading difficulty', 'sentiment', 'toxicity', 'similarity', 'novelty')):
        return 'Text / NLP\nfeatures'
    if 'author' in lower or 'prior author' in lower:
        return 'Author history'
    return 'Timing / discussion\ncontext'

regression_plot['feature_category'] = regression_plot['feature'].map(regression_feature_category)
regression_plot['odds_ratio'] = np.exp(regression_plot['coefficient'])

matched = sorted(regression_plot['outcome'].unique())
outcomes_without_coefficients = sorted(set(FEATURE_LABELS) - set(matched))
print(f'Matched outcome features: {len(matched)}')
print('Unmatched policy outcomes:', outcomes_without_coefficients)
display(regression_plot[['outcome', 'feature_label', 'selector', 'depth', 'forum', 'coefficient']].sort_values(['depth', 'selector', 'outcome']))

display(correlations_by(regression_plot, 'forum', 'coefficient', ['depth', 'selector']))
odds_ratio_correlations = correlations_by(regression_plot, 'forum', 'odds_ratio', ['depth', 'selector'])
print('Odds-ratio correlations (including Spearman rank correlation):')
display(odds_ratio_correlations)
plot_regression_forum_coefficients(regression_plot)

## 5. ML FORUM versus SHAP

Stage 9 calculates SHAP on the sealed `paper2_test` comments using a development-only background. The exported summary averages signed comment attributions within story and then averages stories, matching the FORUM estimand. The `metadata_bge` feature set is labelled as metadata + text, with all BGE dimensions summed into one `text_bge` block. Mean absolute SHAP remains available as an importance diagnostic, but the comparisons below use signed mean SHAP contributions.

In [ ]:
SHAP_SUMMARY = PAPER1_TABLES / 'held_out_shap_importance.parquet'
if not SHAP_SUMMARY.exists():
    raise FileNotFoundError(f'Run notebook 09 before this section: {SHAP_SUMMARY}')

shap_summary = pd.read_parquet(SHAP_SUMMARY)
shap_summary = shap_summary[shap_summary['feature_group'].ne('selector')].copy() if 'feature_group' in shap_summary else shap_summary
shap_summary['model_family_label'] = shap_summary['model_family'].map({'xgboost': 'XGB', 'neural': 'NN'}).fillna(shap_summary['model_family'])

ml_orderings = [
    'xgb_metadata_audience', 'xgb_metadata_editor',
    'xgb_metadata_text_audience', 'xgb_metadata_text_editor',
    'neural_metadata_audience', 'neural_metadata_editor',
    'neural_metadata_text_audience', 'neural_metadata_text_editor',
]
ml_forum = ranking_metrics.query(
    "ordering in @ml_orderings and reply_mode == 'loose' and not pinned"
).copy()
ml_forum['model_family'] = np.where(ml_forum['ordering'].str.startswith('xgb'), 'xgboost', 'neural')
ml_forum['feature_set'] = np.where(ml_forum['ordering'].str.contains('text'), 'metadata_bge', 'metadata')
ml_forum['selector'] = np.where(ml_forum['ordering'].str.contains('audience'), 'audience', 'curator')
ml_forum = ml_forum.groupby(['model_family', 'feature_set', 'selector', 'outcome', 'feature_label', 'depth'], as_index=False).agg(
    forum=('forum', 'mean'), n_policies=('policy_id', 'nunique')
)

ml_shap_forum = ml_forum.merge(
    shap_summary,
    left_on=['model_family', 'feature_set', 'outcome'],
    right_on=['model_family', 'feature_set', 'feature'],
    how='inner',
    suffixes=('', '_shap'),
)
if ml_shap_forum.empty:
    raise RuntimeError('No ML FORUM rows matched the Stage 9 SHAP summary')
ml_shap_forum['shap_value'] = np.where(ml_shap_forum['selector'].eq('audience'), ml_shap_forum['mean_shap_audience'], ml_shap_forum['mean_shap_curator'])
ml_shap_forum['shap_gap'] = ml_shap_forum['mean_shap_gap']
ml_shap_forum['model_variant'] = ml_shap_forum.apply(
    lambda row: model_selector_label(row['model_family'], row['feature_set'], row['selector']),
    axis=1,
)
ml_shap_forum['model_kind'] = ml_shap_forum['model_variant'].str.split(':').str[0]
ml_shap_forum['feature_category'] = ml_shap_forum['feature_label'].map(regression_feature_category)
display(correlations_by(ml_shap_forum, 'forum', 'shap_value', ['model_family', 'feature_set', 'selector', 'depth']))
display(ml_shap_forum.sort_values(['model_family', 'feature_set', 'selector', 'depth', 'outcome']).head(30))

plot_ml_forum_shap(ml_shap_forum)

forum_gap = ml_forum.pivot_table(
    index=['model_family', 'feature_set', 'outcome', 'feature_label', 'depth'],
    columns='selector', values='forum', aggfunc='first'
).reset_index()
forum_gap = forum_gap.rename(columns={'audience': 'forum_audience', 'curator': 'forum_curator'})
forum_gap['forum_gap'] = forum_gap['forum_curator'] - forum_gap['forum_audience']
gap_panel = forum_gap.merge(
    shap_summary,
    left_on=['model_family', 'feature_set', 'outcome'],
    right_on=['model_family', 'feature_set', 'feature'],
    how='inner',
)
display(correlations_by(gap_panel, 'forum_gap', 'mean_shap_gap', ['model_family', 'feature_set', 'depth']))

## Interpretation checklist

When reviewing the plots, prioritise points that are far from the relevant reference pattern, but keep the aggregation and design in view. A high correlation does not imply that FORUM and nDCG are interchangeable; the scientifically interesting cases are ranking conditions where one metric is high and the other is unexpectedly low, or where the top-10/full-list relationship changes sharply by feature.